# Anima + WAI-Anima — ComfyUI Colab

Один ноутбук для Anima Aesthetic v1.1 и WAI-Anima v1.0. Устанавливает ComfyUI, ComfyUI-Manager, Anima-LLLite и готовые T2I/ControlNet/Inpaint workflow. Токены берутся из Colab Secrets (`HF_TOKEN`, `CIVITAI_API_TOKEN`) или запрашиваются интерактивно.

In [ ]:
# @title 1) Tokens (Colab Secrets / environment only)
# Tokens are read from Colab Secrets (panel 🔑) or process environment.
# Handles late "Grant Access": if secret not found immediately we poll up to ~90s
# so user can enable Notebook access ON + Grant Access during first cell.
import os, time

!pip install -q -U huggingface_hub
from huggingface_hub import login

try:
    from google.colab import userdata
except Exception:
    userdata = None


def colab_secret(name: str) -> str:
    """Read a secret from env first, then Colab Secrets."""
    value = os.environ.get(name, "").strip()
    if not value and userdata is not None:
        try:
            raw = userdata.get(name) or ""
        except Exception:
            raw = ""
        if isinstance(raw, dict):
            raw = raw.get("value") or raw.get("token") or next(iter(raw.values()), "")
        value = str(raw).strip()
    return value


HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    print("HF_TOKEN не найден — жду до 90 сек (можешь сейчас нажать Grant Access / включить Notebook access ON)...")
    for _attempt in range(30):
        time.sleep(3)
        HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
        if HF_TOKEN:
            print("✓ HF_TOKEN появился — продолжаю.")
            break
        print(".", end="", flush=True)
    print()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found.\n"
        "  • Colab: открой 🔑 Secrets слева, добавь HF_TOKEN, включи Notebook access ON, нажми Grant Access, затем Runtime → Rerun.\n"
        "  • Если ты нажал Grant Access только что — токен должен был подхватиться за 90 сек; если не подхватился, перезапусти ячейку.\n"
        "  • Jupyter локально: export HF_TOKEN=... перед стартом."
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF auth: OK")

CIVITAI_API_TOKEN = colab_secret("CIVITAI_API_TOKEN")
if CIVITAI_API_TOKEN:
    os.environ["CIVITAI_API_TOKEN"] = CIVITAI_API_TOKEN
    print("Civitai auth: OK")
else:
    print("Civitai token not set (optional); Civitai downloads may return 403.")

print("Done.")


In [ ]:
# @title 2) Install ComfyUI + Managers + node pack (incl. rgthree)
# Idempotent: safe to rerun. Unified template: swap + aria2 split.
import os
import subprocess
import sys

def run(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

# Swap protects small-RAM Colab VMs during dependency resolution.
if not os.path.exists("/swapfile"):
    print("Creating swap (8GB)...")
    try:
        subprocess.run("sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile", shell=True, check=True)
        print("Swap enabled.")
    except Exception as e:
        print(f"Swap creation failed ({e}), continuing.")

# System deps: aria2 + ffmpeg (cloudflared handled in launch cell)
import shutil
if shutil.which("aria2c") is None:
    print("Installing aria2 + ffmpeg...")
    subprocess.run("apt-get -y update -qq", shell=True, check=False)
    subprocess.run("apt-get -y install -qq aria2 ffmpeg || true", shell=True, check=False)
else:
    print("aria2c already present:", shutil.which("aria2c"))

COMFY_ROOT_D = "/content/ComfyUI"
MODEL_ROOT = os.path.join(COMFY_ROOT_D, "models")

if not os.path.exists(COMFY_ROOT_D):
    run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
run(f"pip install -q -r {COMFY_ROOT_D}/requirements.txt")

NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "ComfyUI-Model-Manager": "https://github.com/hayden-cn/ComfyUI-Model-Manager.git#v2.8.4",
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF.git",
    "comfyui-kjnodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI-Workflow-Models-Downloader": "https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git",
    # Anima-specific nodes:
    "ComfyUI-Anima-LLLite": "https://github.com/kohya-ss/ComfyUI-Anima-LLLite.git",
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "comfyui-lora-manager": "https://github.com/willmiao/ComfyUI-Lora-Manager.git",
    "was-node-suite-comfyui": "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    "ComfyUI-Image-Saver": "https://github.com/alexopus/ComfyUI-Image-Saver.git",
}
for folder, repo in NODES.items():
    target = os.path.join(COMFY_ROOT_D, "custom_nodes", folder)
    if not os.path.exists(target):
        repo_url, _, ref = repo.partition("#")
        clone_cmd = ["git", "clone", "--depth", "1"]
        if ref:
            clone_cmd += ["--branch", ref]
        clone_cmd += [repo_url, target]
        print("+", " ".join(clone_cmd))
        subprocess.run(clone_cmd, check=True)
    req = os.path.join(target, "requirements.txt")
    if os.path.exists(req):
        run(f"pip install -q -r {req}")
print("ComfyUI and node set are ready.")


In [ ]:
# @title 3) Download both Anima checkpoints and dependencies
import requests
from pathlib import Path
def download(url, target, headers=None):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>1024: print('exists:',target); return
    h={'Authorization':f'Bearer {HF_TOKEN}'} if 'huggingface.co' in url else {}
    if headers: h.update(headers)
    with requests.get(url,headers=h,stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(target,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    print('downloaded:',target)
def civitai(url,target):
    h={'Authorization':f'Bearer {CIVITAI_API_TOKEN}'} if CIVITAI_API_TOKEN else {}
    download(url,target,h)
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors',Path(MODEL_ROOT)/'text_encoders/qwen_3_06b_base.safetensors')
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors',Path(MODEL_ROOT)/'vae/qwen_image_vae.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-any-test-like-v2.safetensors',Path(MODEL_ROOT)/'model_patches/anima-lllite-any-test-like-v2.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-inpainting-v2.safetensors',Path(MODEL_ROOT)/'model_patches/anima-lllite-inpainting-v2.safetensors')
civitai('https://civitai.red/api/download/models/3126581?fileId=3007030',Path(MODEL_ROOT)/'diffusion_models/anima/anima_aestheticV11.safetensors')
civitai('https://civitai.red/api/download/models/2983680?fileId=2863158',Path(MODEL_ROOT)/'diffusion_models/anima/waiANIMA_v10Base10.safetensors')
print('Both checkpoints are available in ComfyUI.')

In [ ]:
# @title 5) Launch ComfyUI + Cloudflare Quick Tunnel
# Реализация туннеля идентична Hermes Dashboard-ячейке:
#   1. поднимаем ComfyUI локально и ждём готовности;
#   2. находим/скачиваем cloudflared;
#   3. останавливаем старый туннель при повторном запуске;
#   4. поднимаем Quick Tunnel на локальный порт ComfyUI;
#   5. читаем URL из лога, ждём DNS, проверяем публичный доступ.
import base64, os, re, shutil, socket, stat, subprocess, threading, time
import queue
from pathlib import Path

import requests

LOW_VRAM_STABLE = False  # False: --lowvram + Dynamic VRAM (optimal for T4 15GB); True: --novram ultra-low (2.5x slower, risks RAM OOM)
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
OUTPUT_DIR = COMFY_ROOT / "output"
COMFY_PORT = globals().get("COMFY_PORT", 8188)

# ── Model Manager token bridge (keys live in its private.key pickle) ────────
MODEL_MANAGER_DIR = COMFY_ROOT / "custom_nodes" / "ComfyUI-Model-Manager"
if MODEL_MANAGER_DIR.exists():
    import pickle

    manager_key_file = MODEL_MANAGER_DIR / "private.key"
    manager_keys = {}
    if manager_key_file.exists():
        try:
            with manager_key_file.open("rb") as stream:
                loaded = pickle.load(stream)
            if isinstance(loaded, dict):
                manager_keys.update(loaded)
        except Exception:
            print("Existing Model Manager key file was unreadable; recreating it.")

    hf_for_manager = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()
    civitai_for_manager = os.environ.get("CIVITAI_API_TOKEN", "").strip()
    if hf_for_manager:
        manager_keys["huggingface"] = hf_for_manager
    if civitai_for_manager:
        manager_keys["civitai"] = civitai_for_manager
    if manager_keys:
        manager_key_tmp = manager_key_file.with_suffix(".private.key.tmp")
        with manager_key_tmp.open("wb") as stream:
            pickle.dump(manager_keys, stream, protocol=pickle.HIGHEST_PROTOCOL)
        manager_key_tmp.replace(manager_key_file)
    print(
        "Model Manager token bridge:",
        "HF=" + ("yes" if hf_for_manager else "existing/none"),
        "Civitai=" + ("yes" if civitai_for_manager else "existing/none"),
    )
else:
    print("⚠ ComfyUI-Model-Manager not found — run the install cell first.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()


# Safe rerun: stop processes started by any earlier launch-cell version.
for process_name in ("_TUNNEL_PROC", "_COMFY_PROC"):
    stop_process(globals().get(process_name))
old_log = globals().get("_COMFY_LOG")
if old_log is not None:
    try:
        old_log.close()
    except Exception:
        pass


def port_open(host, port, timeout=1):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        return s.connect_ex((host, port)) == 0
    finally:
        s.close()


# ══════════════════════════════════════════════════════════════════════════
# 1. ЗАПУСК COMFYUI ЛОКАЛЬНО
# ══════════════════════════════════════════════════════════════════════════
comfy_args = [
    "python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
    "--enable-cors-header", "*", "--output-directory", str(OUTPUT_DIR),
]
comfy_args += (
    ["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"]
    if LOW_VRAM_STABLE else ["--lowvram", "--preview-method", "auto"]
)
_COMFY_LOG = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
_COMFY_PROC = subprocess.Popen(
    comfy_args, cwd=COMFY_ROOT, stdout=_COMFY_LOG, stderr=subprocess.STDOUT,
)


def local_comfy_ready(timeout=300):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _COMFY_PROC.poll() is not None:
            raise RuntimeError("ComfyUI exited. Inspect /content/comfyui.log")
        try:
            response = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)
            if response.ok:
                return True
        except requests.RequestException:
            time.sleep(2)
    return False


if not local_comfy_ready():
    raise TimeoutError("ComfyUI did not become ready within 300 seconds.")
print("✓ ComfyUI is ready locally on port", COMFY_PORT)


# ══════════════════════════════════════════════════════════════════════════
# 2. НАХОДИМ / СКАЧИВАЕМ CLOUDFLARED  (как в Hermes Dashboard-ячейке)
# ══════════════════════════════════════════════════════════════════════════
cloudflared_candidates = [
    shutil.which("cloudflared"),
    "/usr/local/bin/cloudflared",
]

CLOUDFLARED_BIN = None

for c in cloudflared_candidates:
    if c and Path(c).exists():
        CLOUDFLARED_BIN = Path(c)
        break

if CLOUDFLARED_BIN is None:
    machine = os.uname().machine.lower()

    if machine in ("x86_64", "amd64"):
        cf_arch = "amd64"
    elif machine in ("aarch64", "arm64"):
        cf_arch = "arm64"
    else:
        raise RuntimeError(f"Unknown architecture: {machine}")

    CLOUDFLARED_BIN = Path("/tmp/cloudflared-comfy")
    download_url = (
        "https://github.com/cloudflare/cloudflared/"
        f"releases/latest/download/cloudflared-linux-{cf_arch}"
    )

    print("Скачиваю cloudflared...")
    subprocess.run(
        ["curl", "-fL", "--retry", "3", download_url, "-o", str(CLOUDFLARED_BIN)],
        check=True,
    )
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IXUSR)

print("cloudflared:", CLOUDFLARED_BIN)


# ══════════════════════════════════════════════════════════════════════════
# 3. ПОВТОРНЫЙ ЗАПУСК — ОСТАНАВЛИВАЕМ СТАРЫЙ ТУННЕЛЬ
# ══════════════════════════════════════════════════════════════════════════
old_cf = globals().get("_CLOUDFLARED_PROC")
if old_cf is not None:
    try:
        if old_cf.poll() is None:
            print("Останавливаю предыдущий tunnel...")
            old_cf.terminate()
            try:
                old_cf.wait(timeout=10)
            except Exception:
                old_cf.kill()
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════
# 4. ПОДНИМАЕМ QUICK TUNNEL НА COMFYUI
# ══════════════════════════════════════════════════════════════════════════
print("=" * 78)
print("ЗАПУСК CLOUDFLARE QUICK TUNNEL -> COMFYUI")
print("=" * 78)

_CLOUDFLARED_PROC = subprocess.Popen(
    [
        str(CLOUDFLARED_BIN),
        "tunnel",
        "--no-autoupdate",
        "--url",
        f"http://127.0.0.1:{COMFY_PORT}",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


# ══════════════════════════════════════════════════════════════════════════
# 5. ЧИТАЕМ URL ИЗ ЛОГА
# ══════════════════════════════════════════════════════════════════════════
TUNNEL_URL = None

cf_lines = []

pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

deadline = time.time() + 90

while time.time() < deadline:

    if _CLOUDFLARED_PROC.poll() is not None:
        break

    line = _CLOUDFLARED_PROC.stdout.readline()

    if line:
        cf_lines.append(line.rstrip())
        print(line, end="")

        match = pattern.search(line)

        if match:
            TUNNEL_URL = match.group(0)
            break

    else:
        time.sleep(0.2)

if not TUNNEL_URL:
    print()
    print("=== CLOUDFLARED OUTPUT ===")
    print("\n".join(cf_lines[-200:]))

    # Fallback hint: the next cell exposes ComfyUI through bore.pub instead.
    raise RuntimeError(
        "Не удалось получить Quick Tunnel URL. "
        "Если сеть блокирует trycloudflare — запусти следующую ячейку (bore.pub fallback)."
    )

print()
print("✓ ComfyUI Tunnel:", TUNNEL_URL)


# ══════════════════════════════════════════════════════════════════════════
# 6. ЖДЁМ DNS
# ══════════════════════════════════════════════════════════════════════════
TUNNEL_HOST = TUNNEL_URL.replace("https://", "").split("/")[0]

dns_ok = False

for attempt in range(1, 31):
    try:
        infos = socket.getaddrinfo(TUNNEL_HOST, 443)
        if infos:
            dns_ok = True
            break
    except socket.gaierror:
        pass
    time.sleep(2)

print("DNS:", "OK" if dns_ok else "NOT READY")


# ══════════════════════════════════════════════════════════════════════════
# 7. ПРОВЕРЯЕМ ПУБЛИЧНЫЙ ДОСТУП
# ══════════════════════════════════════════════════════════════════════════
if dns_ok:

    print("=" * 78)
    print("PUBLIC COMFYUI TEST")
    print("=" * 78)

    public_test = subprocess.run(
        [
            "curl", "-sS", "-L",
            "--connect-timeout", "15",
            "--max-time", "60",
            "--retry", "5",
            "--retry-delay", "2",
            "-o", "/dev/null",
            "-w", "%{http_code}",
            TUNNEL_URL + "/system_stats",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=90,
    )

    public_http = public_test.stdout.strip()
    print("Public /system_stats HTTP:", public_http)

    if public_test.stderr:
        print("curl stderr:", public_test.stderr[:3000])

    if public_http not in ("200", "403"):
        print("⚠ Public check inconclusive — tunnel может ещё прогреваться.")

print()
print("=" * 78)
print("✓ COMFYUI ГОТОВ — ОТКРЫВАЙ В БРАУЗЕРЕ:")
print(TUNNEL_URL)
print("=" * 78)
print("Если ссылка даёт 403 — запусти следующую ячейку (bore.pub fallback).")

globals()["TUNNEL_URL"] = TUNNEL_URL

In [ ]:
# @title 5a) Watchdog: Cloudflare Tunnel keepalive (auto-restart, never exits)
# Эта ячейка НЕ завершается — крутится вечно, перезапуская cloudflared при падении.
# Запускай её СРАЗУ после ячейки Cloudflare Tunnel. Останови — Interrupt / Restart Runtime.
import time, re, socket, subprocess, threading
from pathlib import Path
import requests

COMFY_PORT = globals().get("COMFY_PORT", 8188)
CLOUDFLARED_BIN = globals().get("CLOUDFLARED_BIN")
# fallback if re-run without CF cell
if CLOUDFLARED_BIN is None:
    import shutil, os, stat
    for c in [shutil.which("cloudflared"), "/usr/local/bin/cloudflared", "/tmp/cloudflared-comfy"]:
        if c and Path(c).exists():
            CLOUDFLARED_BIN = Path(c)
            break
    if CLOUDFLARED_BIN is None:
        raise RuntimeError("cloudflared not found — сначала запусти ячейку Cloudflare Tunnel (5).")

_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

def _restart_cf():
    old = globals().get("_CLOUDFLARED_PROC")
    if old is not None and old.poll() is None:
        try:
            old.terminate()
            try: old.wait(timeout=5)
            except: old.kill()
        except: pass
    # also drain old stdout thread implicitly
    print(f"[{time.strftime('%H:%M:%S')}] Перезапускаю cloudflared -> 127.0.0.1:{COMFY_PORT} ...")
    proc = subprocess.Popen(
        [str(CLOUDFLARED_BIN), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    globals()["_CLOUDFLARED_PROC"] = proc
    url = None
    deadline = time.time() + 90
    buf = []
    while time.time() < deadline:
        if proc.poll() is not None:
            print("cloudflared exited early, log tail:")
            print("\n".join(buf[-20:]))
            break
        line = proc.stdout.readline()
        if line:
            buf.append(line.rstrip())
            print(line, end="")
            m = _pattern.search(line)
            if m:
                url = m.group(0)
                break
        else:
            time.sleep(0.2)
    if not url:
        print("Не удалось получить новый TUNNEL_URL, пробую снова через 5с...")
        return None
    globals()["TUNNEL_URL"] = url
    print(f"\n✓ Новый TUNNEL_URL: {url}")
    # wait DNS
    host = url.replace("https://","").split("/")[0]
    for _ in range(15):
        try:
            if socket.getaddrinfo(host, 443):
                print("DNS OK:", host)
                break
        except socket.gaierror:
            pass
        time.sleep(2)
    return url

# initial sanity
if globals().get("_COMFY_PROC") is None or globals().get("_COMFY_PROC").poll() is not None:
    print("⚠ ComfyUI не запущен (_COMFY_PROC dead) — туннель всё равно будет рестартоваться, но ComfyUI нужно перезапустить ячейкой 5.")
if globals().get("_CLOUDFLARED_PROC") is None or globals().get("_CLOUDFLARED_PROC").poll() is not None:
    print("CF туннель не активен — пробую поднять...")
    _restart_cf()
else:
    print(f"Watchdog стартовал. Текущий TUNNEL_URL: {globals().get('TUNNEL_URL', '(unknown)')}")
    print(f"ComfyUI PID: {globals().get('_COMFY_PROC').pid if globals().get('_COMFY_PROC') else 'none'} | cloudflared PID: {globals().get('_CLOUDFLARED_PROC').pid}")

_keepalive = 0
try:
    while True:
        time.sleep(5)
        _keepalive += 5
        comfy = globals().get("_COMFY_PROC")
        cf = globals().get("_CLOUDFLARED_PROC")
        tunnel_url = globals().get("TUNNEL_URL", "")
        # ComfyUI health
        if comfy is None or comfy.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] ⚠ ComfyUI упал (exit={comfy.poll() if comfy else 'no proc'}). Проверь /content/comfyui.log . Жду 5с...")
            time.sleep(5)
            continue
        # quick local check every 60s
        if _keepalive >= 60:
            _keepalive = 0
            try:
                r = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)
                local_ok = r.ok
            except Exception as e:
                local_ok = False
            # tunnel process check
            if cf is None or cf.poll() is not None:
                print(f"[{time.strftime('%H:%M:%S')}] ⚠ cloudflared упал (exit={cf.poll() if cf else 'none'}) — перезапуск...")
                new_url = _restart_cf()
                if new_url and local_ok:
                    try:
                        # public check best-effort
                        pr = requests.get(new_url + "/system_stats", timeout=10)
                        print(f"Public check: {pr.status_code}")
                    except Exception as e:
                        print(f"Public check pending: {e}")
                continue
            # also public ping (best effort, 403 is ok)
            if tunnel_url:
                try:
                    pr = requests.get(tunnel_url + "/system_stats", timeout=10)
                    if pr.status_code in (200,403):
                        print(f"[{time.strftime('%H:%M:%S')}] keepalive OK | local={local_ok} public={pr.status_code} | {tunnel_url}")
                    else:
                        print(f"[{time.strftime('%H:%M:%S')}] public HTTP {pr.status_code} — возможно прогрев, туннель жив")
                except Exception as e:
                    print(f"[{time.strftime('%H:%M:%S')}] public ping fail ({e}) — рестартую туннель...")
                    _restart_cf()
            else:
                print(f"[{time.strftime('%H:%M:%S')}] keepalive local={local_ok} — TUNNEL_URL пуст, рестартую...")
                _restart_cf()
        else:
            # fast poll for dead tunnel
            if cf is None or cf.poll() is not None:
                print(f"[{time.strftime('%H:%M:%S')}] ⚠ cloudflared упал — мгновенный перезапуск...")
                _restart_cf()
                _keepalive = 0
except KeyboardInterrupt:
    print("\nWatchdog остановлен (KeyboardInterrupt). Туннель остаётся жив пока процесс не убит.")



In [ ]:
# @title 5b) Fallback: bore.pub tunnel (если Cloudflare URL заблокирован / 403)
# Некоторые сети/страны получают 403 от Cloudflare на *.trycloudflare.com ссылки.
# Эта ячейка открывает ТОТ ЖЕ локальный ComfyUI через bore.pub.
# Обычный HTTP без шифрования и случайный порт — используй только как fallback.
# Останавливает CF-туннель, чтобы не держать два сразу.
import os
import re as _re
import subprocess
import threading as _threading
import time

import requests

COMFY_PORT = globals().get("COMFY_PORT", 8188)

# Останавливаем Cloudflare туннель из предыдущей ячейки (если был).
_old_cf = globals().get("_CLOUDFLARED_PROC")
if _old_cf is not None and _old_cf.poll() is None:
    _old_cf.terminate()
    try:
        _old_cf.wait(timeout=10)
    except Exception:
        _old_cf.kill()
    print("Cloudflare tunnel остановлен.")


def _ensure_bore():
    import shutil

    if shutil.which("bore"):
        return "bore"
    local = "/tmp/bore"
    if os.path.exists(local) and os.access(local, os.X_OK):
        return local
    arch = {"x86_64": "x86_64", "amd64": "x86_64", "aarch64": "aarch64"}.get(
        os.uname().machine.lower(), "x86_64"
    )
    url = (
        "https://github.com/ekzhang/bore/releases/download/v0.5.0/"
        f"bore-v0.5.0-{arch}-unknown-linux-musl.tar.gz"
    )
    print("Скачиваю bore CLI...")
    subprocess.run(["curl", "-fsSL", "-o", "/tmp/bore.tar.gz", url], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/bore.tar.gz", "-C", "/tmp"], check=True)
    os.chmod("/tmp/bore", 0o755)
    return local


BORE_BIN = _ensure_bore()

# Повторный запуск ячейки — останавливаем старый fallback-туннель.
old = globals().get("_BORE_PROC")
if old is not None and old.poll() is None:
    old.terminate()
    try:
        old.wait(timeout=5)
    except Exception:
        old.kill()


# Убеждаемся, что ComfyUI ещё жив (запущен предыдущей ячейкой).
assert globals().get("_COMFY_PROC") is not None and _COMFY_PROC.poll() is None, (
    "ComfyUI не запущен — сначала выполни предыдущую ячейку."
)

_BORE_LOG = open("/content/bore.log", "a", encoding="utf-8", buffering=1)
_BORE_PROC = subprocess.Popen(
    [BORE_BIN, "local", str(COMFY_PORT), "--to", "bore.pub"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

_public_port = None
_deadline = time.time() + 45
_lines = []


def _pump(pipe):
    # Robust: pipe may be None (if Popen failed) or closed; avoid AttributeError
    if pipe is None:
        _lines.append("[bore] no stdout pipe (Popen failed)")
        return
    try:
        for line in iter(pipe.readline, ""):
            if line is None:
                break
            _lines.append(line.rstrip())
            if not line:
                time.sleep(0.1)
    except Exception as e:
        _lines.append(f"[bore pump error] {e}")


_threading.Thread(target=_pump, args=(_BORE_PROC.stdout,), daemon=True).start()

while time.time() < _deadline and _public_port is None:
    if _BORE_PROC.poll() is not None:
        print("bore процесс завершился преждевременно. Лог:")
        print("\n".join(_lines[-20:]))
        break
    for line in list(_lines):
        m = _re.search(r"listening at bore\.pub:(\d+)", line)
        if m:
            _public_port = m.group(1)
            break
    time.sleep(0.5)

if not _public_port:
    print("\n".join(_lines[-20:]) if _lines else "(нет вывода от bore)")
    raise RuntimeError("bore.pub tunnel failed to start. Попробуй перезапустить ячейку или используй Cloudflare tunnel (предыдущая ячейка).")

PUBLIC_URL = f"http://bore.pub:{_public_port}"
print("=" * 78)
print("✓ COMFYUI ЧЕРЕЗ BORE.PUB:", PUBLIC_URL)
print("(plain HTTP — используй только если Cloudflare URL заблокирован)")
print("=" * 78)

for _ in range(6):
    try:
        if requests.get(PUBLIC_URL + "/system_stats", timeout=8).ok:
            print("Public check: OK")
            break
    except requests.RequestException:
        pass
    time.sleep(3)

globals()["PUBLIC_URL"] = PUBLIC_URL


In [ ]:
# @title 5c) Watchdog: bore.pub keepalive (auto-restart, never exits)
# Эта ячейка НЕ завершается — крутится вечно, перезапуская bore при падении.
# Запускай её СРАЗУ после ячейки bore.pub fallback. Останови — Interrupt / Restart Runtime.
import time, re, subprocess, threading, os
from pathlib import Path
import requests

COMFY_PORT = globals().get("COMFY_PORT", 8188)
BORE_BIN = globals().get("BORE_BIN")
if BORE_BIN is None:
    import shutil
    if shutil.which("bore"):
        BORE_BIN = "bore"
    elif Path("/tmp/bore").exists():
        BORE_BIN = "/tmp/bore"
    else:
        raise RuntimeError("bore not found — сначала запусти ячейку bore.pub (5b).")

def _restart_bore():
    old = globals().get("_BORE_PROC")
    if old is not None and old.poll() is None:
        try:
            old.terminate()
            try: old.wait(timeout=5)
            except: old.kill()
        except: pass
    print(f"[{time.strftime('%H:%M:%S')}] Перезапускаю bore local {COMFY_PORT} --to bore.pub ...")
    log = open("/content/bore.log", "a", encoding="utf-8", buffering=1)
    globals()["_BORE_LOG"] = log
    proc = subprocess.Popen(
        [BORE_BIN, "local", str(COMFY_PORT), "--to", "bore.pub"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    globals()["_BORE_PROC"] = proc
    lines=[]
    def _pump(pipe):
        if pipe is None:
            lines.append("[bore] no pipe")
            return
        try:
            for line in iter(pipe.readline, ""):
                if line is None: break
                lines.append(line.rstrip())
                if not line: time.sleep(0.1)
        except Exception as e:
            lines.append(f"[bore pump error] {e}")
    threading.Thread(target=_pump, args=(proc.stdout,), daemon=True).start()
    deadline = time.time() + 45
    port=None
    while time.time() < deadline and port is None:
        if proc.poll() is not None:
            print("bore exited early, log:")
            print("\n".join(lines[-20:]))
            break
        for line in list(lines):
            m = re.search(r"listening at bore\.pub:(\d+)", line)
            if m:
                port=m.group(1)
                break
        time.sleep(0.5)
    if not port:
        print("\n".join(lines[-20:]) if lines else "(no bore output)")
        print("bore restart failed, retry in 5s...")
        return None
    url = f"http://bore.pub:{port}"
    globals()["PUBLIC_URL"] = url
    print(f"✓ Новый PUBLIC_URL: {url}")
    for _ in range(6):
        try:
            if requests.get(url+"/system_stats", timeout=8).ok:
                print("Public check: OK")
                break
        except: pass
        time.sleep(3)
    return url

if globals().get("_COMFY_PROC") is None or globals().get("_COMFY_PROC").poll() is not None:
    print("⚠ ComfyUI не запущен — bore watchdog всё равно стартует, но ComfyUI нужен из ячейки 5.")
if globals().get("_BORE_PROC") is None or globals().get("_BORE_PROC").poll() is not None:
    print("bore туннель не активен — поднимаю...")
    _restart_bore()
else:
    print(f"Watchdog стартовал. PUBLIC_URL: {globals().get('PUBLIC_URL','(unknown)')}")
    print(f"bore PID: {globals().get('_BORE_PROC').pid if globals().get('_BORE_PROC') else 'none'}")

_keepalive=0
try:
    while True:
        time.sleep(5)
        _keepalive+=5
        comfy=globals().get("_COMFY_PROC")
        bore=globals().get("_BORE_PROC")
        pub=globals().get("PUBLIC_URL","")
        if comfy is None or comfy.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] ⚠ ComfyUI упал — жду...")
            time.sleep(5)
            continue
        if bore is None or bore.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] ⚠ bore упал (exit={bore.poll() if bore else 'none'}) — перезапуск...")
            _restart_bore()
            _keepalive=0
            continue
        if _keepalive>=60:
            _keepalive=0
            try:
                r=requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)
                local_ok=r.ok
            except: local_ok=False
            if pub:
                try:
                    pr=requests.get(pub+"/system_stats", timeout=8)
                    print(f"[{time.strftime('%H:%M:%S')}] bore keepalive OK | local={local_ok} public={pr.status_code} | {pub}")
                except Exception as e:
                    print(f"[{time.strftime('%H:%M:%S')}] bore public ping fail ({e}) — рестартую...")
                    _restart_bore()
            else:
                print(f"[{time.strftime('%H:%M:%S')}] bore keepalive local={local_ok} — PUBLIC_URL пуст, рестартую...")
                _restart_bore()
except KeyboardInterrupt:
    print("\nBore watchdog остановлен (KeyboardInterrupt).")

